# Домашнее задание №1: Построение сложной сцены из облаков точек

**Выполнил:** evanatas

1. Создать геометрические примитивы (куб, сфера, цилиндр, конус, тор) и облака точек
   (`sample_points_uniformly`, `sample_points_poisson_disk`).
2. Применить трансформации: перенос, масштаб, поворот.
3. Собрать композиции: снеговик, стол, пирамида из кубов, шестилучевая звезда.
4. Визуализировать сцену в Open3D и Plotly.
5. Раскрасить объекты разными цветами.

In [1]:
import open3d as o3d
import numpy as np
import copy
import plotly.graph_objects as go


def center(pcd):
    """Сдвинуть облако точек в начало координат."""
    pcd.translate(-pcd.get_center())
    return pcd


# своя палитра
COL = {
    "snow":   [0.97, 0.97, 1.00],
    "carrot": [0.95, 0.45, 0.05],
    "coal":   [0.10, 0.10, 0.12],
    "wood":   [0.45, 0.27, 0.12],
    "wood2":  [0.35, 0.20, 0.08],
    "gold":   [0.93, 0.74, 0.12],
    "floor":  [0.83, 0.80, 0.72],
}

In [2]:
# Задание 1. Примитивы и облака точек через sample_points_uniformly
cube     = o3d.geometry.TriangleMesh.create_box(1.0, 1.0, 1.0)
sphere   = o3d.geometry.TriangleMesh.create_sphere(radius=0.7)
cylinder = o3d.geometry.TriangleMesh.create_cylinder(radius=0.4, height=1.6)
cone     = o3d.geometry.TriangleMesh.create_cone(radius=0.5, height=1.2)
torus    = o3d.geometry.TriangleMesh.create_torus(torus_radius=0.8, tube_radius=0.25)

meshes = [cube, sphere, cylinder, cone, torus]
prim_colors = [[0.95,0.40,0.30], [0.30,0.80,0.60], [0.30,0.35,0.85],
               [0.95,0.70,0.20], [0.70,0.30,0.60]]

uniform = []
for k, (m, c) in enumerate(zip(meshes, prim_colors)):
    pcd = m.sample_points_uniformly(number_of_points=1200)
    pcd.paint_uniform_color(c)
    pcd.translate((k * 2.5, 0, 0))      # разносим в ряд
    uniform.append(pcd)

o3d.visualization.draw_geometries(uniform, window_name="Примитивы (uniformly)")

In [3]:
# Задание 1. Те же примитивы через sample_points_poisson_disk (равномернее)
poisson = []
for k, (m, c) in enumerate(zip(meshes, prim_colors)):
    pcd = m.sample_points_poisson_disk(number_of_points=900)
    pcd.paint_uniform_color(c)
    pcd.translate((k * 2.5, 0, 0))
    poisson.append(pcd)

o3d.visualization.draw_geometries(poisson, window_name="Примитивы (poisson disk)")

In [4]:
# Задание 2. Трансформации: перенос, масштаб, поворот
base = [m.sample_points_uniformly(1200) for m in meshes]
for p, c in zip(base, prim_colors):
    p.paint_uniform_color(c)

cube_t = copy.deepcopy(base[0]).translate((-3.5, 0, 0))                       # перенос
sphere_t = copy.deepcopy(base[1]).translate((0, 0, 0))                        # без изменений
cyl_t = copy.deepcopy(base[2]).translate((3.5, 0, 0))
cyl_t.scale(1.6, center=cyl_t.get_center())                                   # масштаб
cone_t = copy.deepcopy(base[3]).translate((0, 3.5, 0))
cone_t.rotate(cone_t.get_rotation_matrix_from_xyz((np.pi/2, 0, 0)), center=cone_t.get_center())  # поворот
torus_t = copy.deepcopy(base[4]).translate((0, -3.5, 0))
torus_t.rotate(torus_t.get_rotation_matrix_from_xyz((0, np.pi/3, 0)), center=torus_t.get_center())

o3d.visualization.draw_geometries([cube_t, sphere_t, cyl_t, cone_t, torus_t],
                                  window_name="Трансформации")

In [5]:
# Задание 3. Снеговик: три сферы + нос-конус + глаза и пуговицы
def ball(radius, n, color, pos):
    p = o3d.geometry.TriangleMesh.create_sphere(radius).sample_points_uniformly(n)
    p.paint_uniform_color(color)
    p.translate(pos)
    return p

snowman  = ball(0.90, 1600, COL["snow"], (0, 0, 0.90))   # низ
snowman += ball(0.60, 1100, COL["snow"], (0, 0, 2.05))   # середина
snowman += ball(0.38,  800, COL["snow"], (0, 0, 2.95))   # голова

nose = o3d.geometry.TriangleMesh.create_cone(0.09, 0.45).sample_points_uniformly(250)
nose.paint_uniform_color(COL["carrot"])
nose.rotate(nose.get_rotation_matrix_from_xyz((-np.pi/2, 0, 0)), center=(0, 0, 0))   # вперёд (+Y)
nose.translate((0, 0.38, 2.98))
snowman += nose

for dy in (-0.14, 0.14):                                  # глаза
    snowman += ball(0.05, 120, COL["coal"], (0.30, dy, 3.05))
for dz in (1.80, 2.05, 2.30):                             # пуговицы
    snowman += ball(0.06, 120, COL["coal"], (0.0, 0.60, dz))

o3d.visualization.draw_geometries([snowman], window_name="Снеговик")

In [6]:
# Задание 3. Стол: круглая столешница (цилиндр) + 4 ножки
top = center(o3d.geometry.TriangleMesh.create_cylinder(radius=1.5, height=0.18).sample_points_uniformly(2500))
top.translate((0, 0, 1.6))
top.paint_uniform_color(COL["wood"])

leg = center(o3d.geometry.TriangleMesh.create_cylinder(0.1, 1.5).sample_points_uniformly(450))
leg.paint_uniform_color(COL["wood2"])

table = top
r = 1.05
for ang in (45, 135, 225, 315):
    x, y = r*np.cos(np.deg2rad(ang)), r*np.sin(np.deg2rad(ang))
    table += copy.deepcopy(leg).translate((x, y, 0.75))

o3d.visualization.draw_geometries([table], window_name="Круглый стол")

In [7]:
# Задание 3. Пирамида из кубов: 4 яруса (4x4, 3x3, 2x2, 1x1), синий градиент
step = 0.6
base_cube = center(o3d.geometry.TriangleMesh.create_box(step, step, step).sample_points_uniformly(350))
grad = [[0.10,0.20,0.55], [0.20,0.45,0.80], [0.45,0.70,0.95], [0.75,0.90,1.00]]

pyramid = o3d.geometry.PointCloud()
for level, side in enumerate([4, 3, 2, 1]):
    z = step/2 + level*step
    for i in range(side):
        for j in range(side):
            x = (i - (side - 1) / 2) * step
            y = (j - (side - 1) / 2) * step
            cube = copy.deepcopy(base_cube).translate((x, y, z))
            cube.paint_uniform_color(grad[level])
            pyramid += cube

o3d.visualization.draw_geometries([pyramid], window_name="Пирамида из кубов")

In [8]:
# Задание 3. Шестилучевая звезда из цилиндров (лучи из центра наружу)
beam = o3d.geometry.TriangleMesh.create_cylinder(radius=0.12, height=2.6).sample_points_uniformly(900)
beam.rotate(beam.get_rotation_matrix_from_xyz((0, np.pi/2, 0)), center=(0, 0, 0))  # положить вдоль X
beam.translate((1.3, 0, 0))                                                        # конец в центре
beam.paint_uniform_color(COL["gold"])

star = o3d.geometry.PointCloud()
for i in range(6):
    R = beam.get_rotation_matrix_from_xyz((0, 0, np.deg2rad(i * 60)))
    star += copy.deepcopy(beam).rotate(R, center=(0, 0, 0))

o3d.visualization.draw_geometries([star], window_name="Шестилучевая звезда")

In [9]:
# Задание 4. Итоговая сцена: композиции по кругу (вдоль осей) на круглой площадке
def at(pcd, angle_deg, radius, scale=1.0):
    p = copy.deepcopy(pcd)
    if scale != 1.0:
        p.scale(scale, center=p.get_center())
    a = np.deg2rad(angle_deg)
    p.translate((radius*np.cos(a), radius*np.sin(a), 0))
    return p

R = 9.0
scene  = at(snowman,  90,  R)          # сверху
scene += at(table,     0,  R)          # справа
scene += at(pyramid, 270,  R)          # снизу
scene += at(star,    180,  R, 0.7)     # слева
scene += copy.deepcopy(torus.sample_points_uniformly(1500)).paint_uniform_color(COL["gold"])  # клумба в центре

# круглая площадка (плоский цилиндр-диск)
floor = center(o3d.geometry.TriangleMesh.create_cylinder(radius=13.0, height=0.1).sample_points_uniformly(9000))
floor.paint_uniform_color(COL["floor"])
scene += floor

o3d.visualization.draw_geometries([scene], window_name="Итоговая сцена")

In [10]:
# Задание 4. Та же сцена в Plotly + сохранение картинки в results/results.png
pts = np.asarray(scene.points)
cols = np.asarray(scene.colors)

fig = go.Figure(go.Scatter3d(
    x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
    mode="markers",
    marker=dict(size=1.8, color=cols),
))
fig.update_layout(
    title_text="Итоговая сцена (Plotly)",
    scene=dict(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
        aspectmode="data",
        camera=dict(eye=dict(x=1.5, y=-1.7, z=1.15)),   # свой ракурс
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    paper_bgcolor="white",
)

import os
os.makedirs("results", exist_ok=True)
fig.write_image("results/results.png", width=1100, height=800, scale=2)
fig.show()